# LeetCode 3739 — Count Subarrays With Majority Element II bằng Python

- Link: https://leetcode.com/problems/count-subarrays-with-majority-element-ii/
- Độ khó: **Hard**
- Chủ đề: Biến đổi `±1`, **tổng tiền tố**, đếm cặp `P[i] < P[j]` (nghịch thế ngược), Fenwick Tree, merge sort, kỹ thuật `O(n)` khi tiền tố chỉ đổi `±1`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countMajoritySubarrays(self, nums: List[int], target: int) -> int:
```

## 1. Mục tiêu học

1. Biến điều kiện "`target` xuất hiện **nhiều hơn một nửa**" thành "tổng `±1` của mảng con **> 0**".
2. Quy về đếm cặp tiền tố `i < j` với `P[i] < P[j]`.
3. Đếm cặp bằng Fenwick Tree (`O(n log n)`) và bằng merge sort.
4. Khai thác tính chất `P` chỉ thay đổi `±1` mỗi bước để đếm trong **`O(n)`**.
5. Kiểm chứng bằng vét cạn.

## 2. Đề bài & ví dụ

Đếm số mảng con của `nums` mà `target` là **phần tử đa số** (xuất hiện **nhiều hơn hẳn** một nửa số phần tử của mảng con).

**Ví dụ 1:** `nums = [1,2,2,3], target = 2` → `5`.

**Ví dụ 2:** `nums = [1,1,1,1], target = 1` → `10`.

**Ví dụ 3:** `nums = [1,2,3], target = 4` → `0`.

**Ràng buộc:** `1 ≤ n ≤ 10^5`, giá trị tới `10^9`.

In [1]:
from typing import List
from collections import defaultdict
import random, time

examples = [([1, 2, 2, 3], 2, 5), ([1, 1, 1, 1], 1, 10), ([1, 2, 3], 4, 0)]
for a, t, exp in examples:
    print(a, t, "-> kỳ vọng", exp)

[1, 2, 2, 3] 2 -> kỳ vọng 5
[1, 1, 1, 1] 1 -> kỳ vọng 10
[1, 2, 3] 4 -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Biến đổi `±1`

Đặt `b[i] = +1` nếu `nums[i] = target`, ngược lại `−1`. Trong mảng con dài `L` có `c` lần `target`: tổng `b` = `c − (L − c) = 2c − L`. `c > L/2` ⇔ `2c − L > 0` ⇔ **tổng > 0**.

In [2]:
nums, t = [1, 2, 2, 3], 2
b = [1 if x == t else -1 for x in nums]
P = [0]
for v in b: P.append(P[-1] + v)
print("b =", b, "| tiền tố P =", P)

b = [-1, 1, 1, -1] | tiền tố P = [0, -1, 0, 1, 0]


### 3.2. Tổng đoạn bằng hiệu tiền tố

Tổng `b[i..j−1] = P[j] − P[i]`. Mảng con hợp lệ ⇔ `P[j] > P[i]` với `0 ≤ i < j ≤ n`. Đáp án = số cặp `(i, j)` như vậy.

In [3]:
print("số cặp i<j có P[i] < P[j]:", sum(1 for i in range(len(P)) for j in range(i + 1, len(P)) if P[i] < P[j]))

số cặp i<j có P[i] < P[j]: 5


### 3.3. Fenwick Tree đếm "số phần tử nhỏ hơn"

`P` nằm trong `[−n, n]` ⇒ dịch thành chỉ số `P + n + 1 ∈ [1, 2n+1]`, không cần nén toạ độ.

### 3.4. Vì sao bước `±1` cho phép đếm `O(n)`?

Với Fenwick, mỗi bước ta hỏi "có bao nhiêu tiền tố `< v`" cho một `v` **bất kỳ** ⇒ cần cấu trúc dữ liệu. Nhưng ở đây `v` mới chỉ lệch `±1` so với `v` cũ, nên câu trả lời mới chỉ khác câu trả lời cũ đúng bằng số tiền tố có **một** giá trị cụ thể (`cnt[cur]` hoặc `cnt[cur−1]`). Đây là kỹ thuật "con trỏ trượt trên miền giá trị" — giống thuật toán Mo nhưng con trỏ chỉ dịch 1 đơn vị mỗi bước.

Bảng dưới minh hoạ `P` và tần suất cho `[1,2,2,3]`, `target = 2`:

In [4]:
P = [0]
for x in [1, 2, 2, 3]: P.append(P[-1] + (1 if x == 2 else -1))
for j in range(1, len(P)):
    smaller = [i for i in range(j) if P[i] < P[j]]
    print(f"j={j}: P[j]={P[j]:>2} | các i<j có P[i]<P[j]: {smaller} -> {len(smaller)} mảng con hợp lệ kết thúc tại {j-1}")

j=1: P[j]=-1 | các i<j có P[i]<P[j]: [] -> 0 mảng con hợp lệ kết thúc tại 0
j=2: P[j]= 0 | các i<j có P[i]<P[j]: [1] -> 1 mảng con hợp lệ kết thúc tại 1
j=3: P[j]= 1 | các i<j có P[i]<P[j]: [0, 1, 2] -> 3 mảng con hợp lệ kết thúc tại 2
j=4: P[j]= 0 | các i<j có P[i]<P[j]: [1] -> 1 mảng con hợp lệ kết thúc tại 3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Duyệt `j` từ trái sang phải, cần biết **có bao nhiêu tiền tố trước đó nhỏ hơn `P[j]`**. Vì `P[j] = P[j−1] ± 1`:

- nếu tăng lên `cur + 1`: các tiền tố cũ "nhỏ hơn" vẫn nhỏ hơn, cộng thêm các tiền tố **bằng** `cur` ⇒ `less += cnt[cur]`;
- nếu giảm xuống `cur − 1`: các tiền tố **bằng** `cur − 1` không còn nhỏ hơn ⇒ `less −= cnt[cur − 1]`.

Mỗi bước `O(1)` ⇒ tổng `O(n)`.

## 5. Cách 1 — Brute Force

In [5]:
def brute(nums, target):
    n = len(nums); res = 0
    for i in range(n):
        c = 0
        for j in range(i, n):
            c += nums[j] == target
            if 2 * c > j - i + 1:
                res += 1
    return res

for a, t, exp in examples:
    print(brute(a, t), "kỳ vọng", exp)

5 kỳ vọng 5
10 kỳ vọng 10
0 kỳ vọng 0


**Nút thắt:** `O(n²) = 10^10` với `n = 10^5`.

## 6. Ý tưởng thuật toán / bước đột phá

Đáp án = `#{(i, j) : i < j, P[i] < P[j]}` — chính là số "cặp thuận" của dãy tiền tố (đối của nghịch thế, không tính cặp bằng nhau). Có ba công cụ:

1. Fenwick Tree trên miền giá trị `[−n, n]`.
2. Merge sort đếm cặp `P[i] < P[j]`.
3. Đếm `O(n)` dựa trên bước nhảy `±1` (mục 4).

**Liên hệ với đếm nghịch thế.** Nếu đảo ngược thứ tự so sánh, số cặp `i < j` có `P[i] > P[j]` là **số nghịch thế** của dãy tiền tố. Tổng số cặp là `C(n+1, 2)`; vậy `#{P[i] < P[j]} = C(n+1, 2) − #nghịch thế − #{P[i] = P[j]}`. Ô dưới kiểm tra đẳng thức này trên một ví dụ.

In [6]:
P = [0]
for x in [3, 3, 1, 3, 2, 3]: P.append(P[-1] + (1 if x == 3 else -1))
m = len(P); pairs = m * (m - 1) // 2
inv = sum(1 for i in range(m) for j in range(i + 1, m) if P[i] > P[j])
eq = sum(1 for i in range(m) for j in range(i + 1, m) if P[i] == P[j])
print("C(n+1,2) =", pairs, "| nghịch thế =", inv, "| bằng nhau =", eq, "| cặp thuận =", pairs - inv - eq)

C(n+1,2) = 21 | nghịch thế = 3 | bằng nhau = 6 | cặp thuận = 12


## 7. Các cách tiếp theo

- **Cách 2:** Fenwick Tree — `O(n log n)`.
- **Cách 3:** merge sort đếm cặp thuận — `O(n log n)`.
- **Cách 4:** đếm `O(n)` với `cnt` và `less` (lời giải nộp).

### 7.1. Cách 2 — Fenwick Tree

In [7]:
def fenwick(nums, target):
    n = len(nums); size = 2 * n + 2
    c = [0] * (size + 1)
    def add(i):
        while i <= size: c[i] += 1; i += i & -i
    def prefix(i):
        s = 0
        while i > 0: s += c[i]; i -= i & -i
        return s
    off = n + 1; cur = 0; res = 0
    add(cur + off)
    for x in nums:
        cur += 1 if x == target else -1
        res += prefix(cur + off - 1)               # số tiền tố trước đó < cur
        add(cur + off)
    return res

for a, t, exp in examples:
    print(fenwick(a, t), "kỳ vọng", exp)

5 kỳ vọng 5
10 kỳ vọng 10
0 kỳ vọng 0


### 7.2. Cách 3 — Merge sort đếm cặp thuận

In [8]:
def merge_count(nums, target):
    P = [0]
    for x in nums: P.append(P[-1] + (1 if x == target else -1))
    def sort(a):
        if len(a) <= 1: return a, 0
        m = len(a) // 2
        L, x = sort(a[:m]); R, y = sort(a[m:])
        cnt = x + y; i = 0
        for r in R:                                   # với mỗi r bên phải, đếm số l bên trái < r
            while i < len(L) and L[i] < r: i += 1
            cnt += i
        merged = []; i = j = 0
        while i < len(L) and j < len(R):
            if L[i] <= R[j]: merged.append(L[i]); i += 1
            else: merged.append(R[j]); j += 1
        return merged + L[i:] + R[j:], cnt
    return sort(P)[1]

for a, t, exp in examples:
    print(merge_count(a, t), "kỳ vọng", exp)

5 kỳ vọng 5
10 kỳ vọng 10
0 kỳ vọng 0


### 7.3. Cách 4 — Đếm `O(n)` (lời giải chính)

In [9]:
def linear(nums, target):
    cnt = defaultdict(int); cnt[0] = 1
    cur = less = res = 0                     # less = số tiền tố đã thấy < cur
    for x in nums:
        if x == target:
            less += cnt[cur]; cur += 1
        else:
            cur -= 1; less -= cnt[cur]
        res += less
        cnt[cur] += 1
    return res

for a, t, exp in examples:
    print(linear(a, t), "kỳ vọng", exp)

5 kỳ vọng 5
10 kỳ vọng 10
0 kỳ vọng 0


## 8. TRACE từng bước

In [10]:
def trace_linear(nums, target):
    cnt = defaultdict(int); cnt[0] = 1; cur = less = res = 0
    print(f"bắt đầu: cur=0, cnt={dict(cnt)}")
    for j, x in enumerate(nums):
        if x == target:
            d = cnt[cur]; less += d; cur += 1; note = f"+1: less += cnt[{cur-1}]={d}"
        else:
            cur -= 1; d = cnt[cur]; less -= d; note = f"-1: less -= cnt[{cur}]={d}"
        res += less; cnt[cur] += 1
        print(f"j={j} x={x} {note:<22} -> cur={cur:>2}, less={less} (mảng con hợp lệ kết thúc tại j), res={res}")
    return res

trace_linear([1, 2, 2, 3], 2)

bắt đầu: cur=0, cnt={0: 1}
j=0 x=1 -1: less -= cnt[-1]=0  -> cur=-1, less=0 (mảng con hợp lệ kết thúc tại j), res=0
j=1 x=2 +1: less += cnt[-1]=1  -> cur= 0, less=1 (mảng con hợp lệ kết thúc tại j), res=1
j=2 x=2 +1: less += cnt[0]=2   -> cur= 1, less=3 (mảng con hợp lệ kết thúc tại j), res=4
j=3 x=3 -1: less -= cnt[0]=2   -> cur= 0, less=1 (mảng con hợp lệ kết thúc tại j), res=5


5

In [11]:
trace_linear([3, 3, 1, 3, 2, 3], 3)
print("brute:", brute([3, 3, 1, 3, 2, 3], 3))

bắt đầu: cur=0, cnt={0: 1}
j=0 x=3 +1: less += cnt[0]=1   -> cur= 1, less=1 (mảng con hợp lệ kết thúc tại j), res=1
j=1 x=3 +1: less += cnt[1]=1   -> cur= 2, less=2 (mảng con hợp lệ kết thúc tại j), res=3
j=2 x=1 -1: less -= cnt[1]=1   -> cur= 1, less=1 (mảng con hợp lệ kết thúc tại j), res=4
j=3 x=3 +1: less += cnt[1]=2   -> cur= 2, less=3 (mảng con hợp lệ kết thúc tại j), res=7
j=4 x=2 -1: less -= cnt[1]=2   -> cur= 1, less=1 (mảng con hợp lệ kết thúc tại j), res=8
j=5 x=3 +1: less += cnt[1]=3   -> cur= 2, less=4 (mảng con hợp lệ kết thúc tại j), res=12
brute: 12


## 9. Code nộp LeetCode

In [12]:
class Solution:
    def countMajoritySubarrays(self, nums: List[int], target: int) -> int:
        n = len(nums)
        cnt = [0] * (2 * n + 3)
        off = n + 1
        cur = off                      # tiền tố 0 dịch thành chỉ số off
        cnt[cur] = 1
        less = res = 0
        for x in nums:
            if x == target:
                less += cnt[cur]
                cur += 1
            else:
                cur -= 1
                less -= cnt[cur]
            res += less
            cnt[cur] += 1
        return res

sol = Solution()
for a, t, exp in examples:
    got = sol.countMajoritySubarrays(a, t)
    print(a, t, "->", got, "✔" if got == exp else "✘")

[1, 2, 2, 3] 2 -> 5 ✔
[1, 1, 1, 1] 1 -> 10 ✔
[1, 2, 3] 4 -> 0 ✔


## 10. Vì sao thuật toán đúng

1. **Biến đổi:** mảng con `[i, j−1]` hợp lệ ⇔ `P[j] − P[i] > 0` (mục 3.1–3.2) ⇒ đáp án = `Σ_j #{i < j : P[i] < P[j]}`.
2. **Bất biến của Cách 4:** trước khi xét phần tử thứ `j` (đã thêm `P[0..j−1]` vào `cnt`), `less = #{i ≤ j−1 : P[i] < cur}` với `cur = P[j−1]`.
   - Nếu `P[j] = cur + 1`: `#{P[i] < cur + 1} = #{P[i] < cur} + #{P[i] = cur}` ⇒ `less += cnt[cur]`.
   - Nếu `P[j] = cur − 1`: `#{P[i] < cur − 1} = #{P[i] < cur} − #{P[i] = cur − 1}` ⇒ `less −= cnt[cur − 1]`.
   Sau đó `less` đúng bằng số `i < j` với `P[i] < P[j]`; cộng vào `res`, rồi thêm `P[j]` vào `cnt` — lúc này `P[j]` không `< P[j]` nên bất biến giữ nguyên cho bước sau.
3. Mỗi giá trị `P` nằm trong `[−n, n]` ⇒ mảng `cnt` dịch `n + 1` đủ chỗ.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(n²)` | `O(1)` |
| 2. Fenwick | `O(n log n)` | `O(n)` |
| 3. Merge sort | `O(n log n)` | `O(n)` |
| 4. Đếm `±1` | `O(n)` | `O(n)` |

Đáp án tối đa `n(n+1)/2 ≈ 5·10^9` — Python không tràn số.

## 12. Bộ test

In [13]:
METHODS = [brute, fenwick, merge_count, linear, lambda a, t: Solution().countMajoritySubarrays(a, t)]
for a, t, exp in examples:
    assert all(f(a, t) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [14]:
# 12.2 ca biên
assert Solution().countMajoritySubarrays([5], 5) == 1
assert Solution().countMajoritySubarrays([5], 6) == 0
assert Solution().countMajoritySubarrays([1, 2], 1) == 1          # [1,2]: đúng một nửa, KHÔNG đa số
assert Solution().countMajoritySubarrays([2, 1, 2], 2) == 3 == brute([2, 1, 2], 2)   # [2], [2], [2,1,2]
print("Ca biên: OK")

Ca biên: OK


In [15]:
# 12.3 ngẫu nhiên
rng = random.Random(3739)
for _ in range(2000):
    n = rng.randint(1, 14); a = [rng.randint(1, 3) for _ in range(n)]; t = rng.randint(1, 4)
    r = [f(a, t) for f in METHODS]
    assert len(set(r)) == 1, (a, t, r)
print("2000 test ngẫu nhiên: OK")

2000 test ngẫu nhiên: OK


In [16]:
# 12.4 test lớn
rng = random.Random(1)
for p in (0.3, 0.5, 1.0):
    a = [7 if rng.random() < p else rng.randint(1, 10**9) for _ in range(10**5)]
    t0 = time.time(); x = Solution().countMajoritySubarrays(a, 7); t1 = time.time()
    print(f"tỉ lệ target {p}: đáp án {x}, thời gian {t1 - t0:.3f}s, khớp Fenwick: {x == fenwick(a, 7)}")

tỉ lệ target 0.3: đáp án 197196, thời gian 0.012s, khớp Fenwick: True


tỉ lệ target 0.5: đáp án 1418469066, thời gian 0.012s, khớp Fenwick: True
tỉ lệ target 1.0: đáp án 5000050000, thời gian 0.010s, khớp Fenwick: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `≥` (tính cả trường hợp đúng một nửa)

In [17]:
def wrong_half(nums, target):
    P = [0]
    for x in nums: P.append(P[-1] + (1 if x == target else -1))
    return sum(1 for i in range(len(P)) for j in range(i + 1, len(P)) if P[i] <= P[j])   # SAI

print("sai:", wrong_half([1, 2], 1), "| đúng:", brute([1, 2], 1))

sai: 2 | đúng: 1


### Lỗi 2 — Cập nhật `less` sai thứ tự khi giảm

Khi giảm, phải trừ `cnt` của giá trị **mới** `cur − 1`, không phải giá trị cũ.

In [18]:
def wrong_down(nums, target):
    cnt = defaultdict(int); cnt[0] = 1; cur = less = res = 0
    for x in nums:
        if x == target: less += cnt[cur]; cur += 1
        else: less -= cnt[cur]; cur -= 1                    # SAI: trừ cnt của giá trị cũ
        res += less; cnt[cur] += 1
    return res

print("sai:", wrong_down([1, 2, 2, 3], 2), "| đúng:", brute([1, 2, 2, 3], 2))

sai: 2 | đúng: 5


### Lỗi 3 — Quên tiền tố rỗng `P[0] = 0`

Thiếu `P[0]` ⇒ bỏ sót mọi mảng con bắt đầu từ chỉ số 0.

In [19]:
def wrong_no_p0(nums, target):
    cnt = defaultdict(int); cur = less = res = 0                   # SAI: thiếu cnt[0] = 1
    for x in nums:
        if x == target: less += cnt[cur]; cur += 1
        else: cur -= 1; less -= cnt[cur]
        res += less; cnt[cur] += 1
    return res

print("sai:", wrong_no_p0([1, 1, 1, 1], 1), "| đúng:", brute([1, 1, 1, 1], 1))

sai: 6 | đúng: 10


### Lỗi 4 — Dùng thuật toán Boyer–Moore / đếm theo "phần tử đa số của cả mảng"

Boyer–Moore chỉ tìm ứng viên đa số của **một** mảng, không đếm được số mảng con. Bài này cố định `target` nên chỉ cần biến đổi `±1`.

### Lỗi 5 — Dùng `dict` mà quên giá trị mặc định

`cnt[cur]` với `dict` thường ném `KeyError` khi `cur` chưa xuất hiện (ví dụ lần đầu đi xuống âm). Dùng `defaultdict(int)`, `cnt.get(cur, 0)` hoặc mảng dịch chỉ số `n + 1` như lời giải nộp.

In [20]:
try:
    cnt = {0: 1}; cur = 0; cur -= 1; _ = cnt[cur]
except KeyError as e:
    print("KeyError với khoá", e)

KeyError với khoá -1


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Phần tử `x` chiếm đa số / nhiều hơn …" ⇒ `±1` + tiền tố ⇒ đếm cặp `P[i] < P[j]`.
- Tiền tố chỉ đổi `±1` ⇒ đếm `O(n)` bằng mảng tần suất và một biến "số nhỏ hơn".
- Tiền tố bị chặn trong `[−n, n]` ⇒ BIT không cần nén.
- Cặp bằng nhau là "đúng một nửa" ⇒ không tính.

## 15. Biến thể của bài

1. **Bản I** (`n ≤ 1000`): vét cạn `O(n²)`.
2. **Ít nhất một nửa** (`≥`): đếm `P[i] ≤ P[j]`.
3. **Chiếm hơn `p/q`**: trọng số `+(q−p)` / `−p` ⇒ tiền tố không còn bước `±1` ⇒ dùng BIT/merge sort.
4. **Đếm theo mọi target**: tổng theo từng giá trị, mỗi giá trị xử lý trên các đoạn liên quan (kỹ thuật "chỉ xét lân cận của các lần xuất hiện").

In [21]:
# Biến thể 3: target chiếm hơn p/q độ dài — trọng số (q-p) cho target, -p cho phần tử khác
def count_fraction(nums, target, p, q):
    P = [0]
    for x in nums: P.append(P[-1] + ((q - p) if x == target else -p))
    vals = sorted(set(P)); rk = {v: i + 1 for i, v in enumerate(vals)}; m = len(vals)
    c = [0] * (m + 1); res = 0
    for v in P:
        i = rk[v] - 1; s = 0
        while i > 0: s += c[i]; i -= i & -i
        res += s; i = rk[v]
        while i <= m: c[i] += 1; i += i & -i
    return res

def brute_fraction(nums, target, p, q):
    n = len(nums); r = 0
    for i in range(n):
        cc = 0
        for j in range(i, n):
            cc += nums[j] == target
            if q * cc > p * (j - i + 1): r += 1
    return r

rng = random.Random(15)
for _ in range(400):
    a = [rng.randint(1, 3) for _ in range(rng.randint(1, 12))]; q = rng.randint(2, 5); p = rng.randint(1, q - 1)
    assert count_fraction(a, 1, p, q) == brute_fraction(a, 1, p, q)
assert count_fraction([1, 2, 2, 3], 2, 1, 2) == 5
print("Biến thể 3 OK")

Biến thể 3 OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. Vét cạn | Mọi mảng con | `O(n²)` | Bản I, kiểm chứng |
| 2. Fenwick | Đếm số tiền tố nhỏ hơn | `O(n log n)` | Tổng quát (trọng số bất kỳ) |
| 3. Merge sort | Cặp thuận | `O(n log n)` | Không cần cấu trúc dữ liệu |
| 4. Đếm `±1` | Cập nhật `less` | `O(n)` | **Lời giải chuẩn** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `countMajoritySubarrays([2,2,1,1], 2)` = ?

**Bài 2 (tính tay).** Trace `less` cho `[1,3,1]`, `target = 1`.

**Bài 3 (điền TODO).** Viết `count_at_least_half(nums, target)` (xuất hiện **ít nhất** một nửa).

**Bài 4 (tự viết lại).** Viết lại Cách 4 bằng `dict` và chỉ một biến `cur` (không dùng mảng dịch chỉ số).

**Bài 5 (mini challenge).** Tính **tổng độ dài** của mọi mảng con mà `target` là đa số.

In [22]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def count_at_least_half(nums, target):
    # TODO: đếm P[i] <= P[j]
    pass

In [23]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def total_length(nums, target):
    # TODO: Σ (j - i) trên các cặp P[i] < P[j]; duy trì thêm tổng chỉ số
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `P = [0,1,2,1,0]`. (2) Cặp `P[i] < P[j]`: `(0,1),(0,2),(0,3),(1,2)`. (3) `4`.

**Bài 2.** (1) `x=1`: `less += cnt[0]=1` ⇒ `1`. (2) `x=3`: `cur=0`, `less −= cnt[0]=1` ⇒ `0`. (3) `x=1`: `less += cnt[0]=2` ⇒ `2`; tổng `1+0+2 = 3`.

**Bài 3.** (1) `≤` thay `<`. (2) `less_eq` = số `≤ cur`. (3) Khi tăng: `less_eq += cnt[cur+1]`; khi giảm: `less_eq −= cnt[cur]` (giá trị cũ).

**Bài 4.** (1) `cnt = {0: 1}`. (2) `cnt.get`. (3) So kết quả.

**Bài 5.** (1) Ngoài `cnt[v]`, lưu `sidx[v]` = tổng chỉ số các tiền tố có giá trị `v`. (2) Duy trì `less_sum` song song với `less`. (3) Cộng `less·j − less_sum`.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2
assert Solution().countMajoritySubarrays([2, 2, 1, 1], 2) == 4 == brute([2, 2, 1, 1], 2)
assert Solution().countMajoritySubarrays([1, 3, 1], 1) == 3
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [25]:
# Bài 3, 4
def count_at_least_half_sol(nums, target):
    cnt = defaultdict(int); cnt[0] = 1; cur = le = res = 0      # le = số tiền tố đã thấy <= cur
    le = 1
    for x in nums:
        if x == target:
            cur += 1; le += cnt[cur]
        else:
            le -= cnt[cur]; cur -= 1
        res += le; cnt[cur] += 1; le += 1                      # P[j] tự nó <= cur cho bước sau
    return res

def linear_dict(nums, target):
    cnt = {0: 1}; cur = less = res = 0
    for x in nums:
        if x == target: less += cnt.get(cur, 0); cur += 1
        else: cur -= 1; less -= cnt.get(cur, 0)
        res += less; cnt[cur] = cnt.get(cur, 0) + 1
    return res

def brute_half(nums, target):
    n = len(nums); r = 0
    for i in range(n):
        c = 0
        for j in range(i, n):
            c += nums[j] == target
            if 2 * c >= j - i + 1: r += 1
    return r

rng = random.Random(4)
for _ in range(800):
    a = [rng.randint(1, 3) for _ in range(rng.randint(1, 12))]
    assert count_at_least_half_sol(a, 1) == brute_half(a, 1), a
    assert linear_dict(a, 1) == brute(a, 1)
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [26]:
# Bài 5: tổng độ dài
def total_length_sol(nums, target):
    cnt = defaultdict(int); sidx = defaultdict(int); cnt[0] = 1; sidx[0] = 0
    cur = less = less_sum = res = 0
    for j, x in enumerate(nums, 1):
        if x == target:
            less += cnt[cur]; less_sum += sidx[cur]; cur += 1
        else:
            cur -= 1; less -= cnt[cur]; less_sum -= sidx[cur]
        res += less * j - less_sum
        cnt[cur] += 1; sidx[cur] += j
    return res

def brute_len(nums, target):
    n = len(nums); r = 0
    for i in range(n):
        c = 0
        for j in range(i, n):
            c += nums[j] == target
            if 2 * c > j - i + 1: r += j - i + 1
    return r

rng = random.Random(5)
for _ in range(800):
    a = [rng.randint(1, 3) for _ in range(rng.randint(1, 12))]
    assert total_length_sol(a, 1) == brute_len(a, 1)
print("Bài 5 OK; ví dụ 1:", total_length_sol([1, 2, 2, 3], 2))

Bài 5 OK; ví dụ 1: 10


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
cnt = {0: 1}; cur = less = res = 0       # less = #tiền tố đã thấy < cur
for x in nums:
    if x == target: less += cnt[cur]; cur += 1
    else:           cur -= 1; less -= cnt[cur]
    res += less; cnt[cur] += 1
```

**Checklist:**

- [ ] Đa số là **nghiêm ngặt** (`> 1/2`)?
- [ ] Có tiền tố rỗng `P[0] = 0`?
- [ ] Khi giảm: trừ `cnt` của giá trị **mới**?
- [ ] Thêm `P[j]` vào `cnt` **sau** khi cộng `less`?
- [ ] Đã so với vét cạn?